In [ ]:
load_ext jupyter_black

In [ ]:
from copy import deepcopy
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS
import statsmodels.formula.api as smf
from sklearn.preprocessing import minmax_scale
import pyfixest as pf

from dim_erasure import binarize_df

In [ ]:
options = {
    "q_50": ["1", "2", "3", "4"],
    "q_51": ["A", "B"],
    "q_52": ["1", "2", "3"],
    "q_53": ["A", "B", "C"],
    "q_54": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_55": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_56": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_57": ["1", "2", "3", "4"],
    "q_58": [
        "1,1",
        "1,2",
        "1,3",
        "1,4",
        "2,1",
        "2,2",
        "2,3",
        "2,4",
        "3,1",
        "3,2",
        "3,3",
        "3,4",
        "4,1",
        "4,2",
        "4,3",
        "4,4",
    ],
    "q_59": [
        "Good manners",
        "Independence",
        "Hard work",
        "Feeling of responsibility",
        "Imagination",
        "Tolerance and respect for other people",
        "Thrift, saving money and things",
        "Determination",
        "Religious faith",
        "Not being selfish",
        "Obedience",
    ],
}

id_col = {
    "prism": "conversation_id",
    "chen": "text_id",
    "cad_en": "conversation_id",
    "cad_fr": "conversation_id",
    "cad_pt": "conversation_id",
    "cad_it": "conversation_id",
}

demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [ ]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
questions_baseline_answers = questions[["q_id", "baseline_answer", "domain"]]

domain_qid_map = {
    domain: questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == domain, "q_id"
    ].tolist()
    for domain in domains
}

questions_baseline_answers.loc[
    questions_baseline_answers["domain"] == "salary",
    "baseline_answer",
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == "salary",
        "baseline_answer",
    ]
    .str.replace(",", "")
    .str.extract(r"^[^\d]*(\d+)", expand=False)
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_53", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_53", "baseline_answer"
    ]
    .str.extract(f"({'|'.join(options['q_53'])})", expand=False)
    .replace({"A": 0, "B": 0.5, "C": 1})
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_51", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_51", "baseline_answer"
    ]
    .str.extract(f"({'|'.join(options['q_51'])})", expand=False)
    .replace({"A": 0, "B": 1})
    .astype(float)
)
questions_baseline_answers.loc[
    questions_baseline_answers["q_id"] == "q_58", "baseline_answer"
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == "q_58", "baseline_answer"
    ]
    .str.replace(" ", "")
    .str.extract(f"({'|'.join(options['q_58'])})", expand=False)
    .replace(
        {
            "1,1": 0,
            "1,3": 0,
            "3,1": 0,
            "3,3": 0,
            "2,2": 1,
            "1,2": 0.5,
            "1,4": 0.5,
            "2,3": 0.5,
            "2,1": 0.5,
            "3,2": 0.5,
            "3,4": 0.5,
            "4,1": 0.5,
            "4,3": 0.5,
            "2,4": 1,
            "4,2": 1,
            "4,4": 1,
        }
    )
    .astype(float)
)
for v in options["q_59"]:
    questions_baseline_answers = pd.concat(
        [
            questions_baseline_answers,
            pd.DataFrame(
                {
                    "q_id": f"q_59_{v.replace(' ','').replace(',','')}",
                    "baseline_answer": questions_baseline_answers.loc[
                        questions_baseline_answers["q_id"] == "q_59", "baseline_answer"
                    ]
                    .str.contains(v)
                    .astype(float),
                }
            ),
        ],
        ignore_index=True,
    )
for c in ["q_50", "q_52", "q_54", "q_55", "q_56", "q_57"]:
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"] == c, "baseline_answer"
    ] = (
        questions_baseline_answers.loc[
            questions_baseline_answers["q_id"] == c, "baseline_answer"
        ]
        .str.extract(f"({'|'.join(options[c])})", expand=False)
        .astype(float)
    )

questions_baseline_answers = pd.Series(
    questions_baseline_answers.baseline_answer.values,
    index=questions_baseline_answers.q_id,
).to_dict()

In [ ]:
for dataset in [
    "cad_en",  # "chen",
    "prism",
]:
    all_cols = deepcopy(demographics[dataset])
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})

    for c in [qid for d in domains for qid in domain_qid_map[d] if d != "salary"]:
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])

    for c in domain_qid_map["salary"]:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

    for domain in domains:
        df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
        if domain != "salary":
            df[domain] = df[domain] * 100

    for c in [
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
        "q_59",
    ]:
        if c == "q_53":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 0.5, "C": 1})
            df[c] = df[c].astype(float)
        elif c == "q_51":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 1})
            df[c] = df[c].astype(float)
        elif c == "q_58":
            df[c] = df[c].str.replace(" ", "")
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace(
                {
                    "1,1": 0,
                    "1,3": 0,
                    "3,1": 0,
                    "3,3": 0,
                    "2,2": 1,
                    "1,2": 0.5,
                    "1,4": 0.5,
                    "2,3": 0.5,
                    "2,1": 0.5,
                    "3,2": 0.5,
                    "3,4": 0.5,
                    "4,1": 0.5,
                    "4,3": 0.5,
                    "2,4": 1,
                    "4,2": 1,
                    "4,4": 1,
                }
            )
            df[c] = df[c].astype(float)
        elif c == "q_59":
            for v in options[c]:
                df[f"{c}_{v.replace(' ','').replace(',','')}"] = (
                    df[c].str.contains(v).astype(float)
                )
        else:
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].astype(float)
    df = df.drop(
        columns=[f"q_{i}" for i in range(50)]
        + ["q_59", "q_60"]
        + [f"q_{i}" for i in range(61, 211)]
    )

    df_linguistic = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    ).drop(
        columns=[
            "s_neutral_model_response",
            "s_neutral_user_prompt",
            "model_response_liwc_Segment",
            "user_prompt_liwc_Segment",
        ],
        errors="ignore",
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df_linguistic:
            df_linguistic[c] = df_linguistic[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    if dataset != "chen":
        if dataset == "prism":
            all_cols += ["model_name"]
        all_cols += ["topic"]
        group_cols = [id_col[dataset]] + all_cols
        df_linguistic = (
            df_linguistic.groupby(group_cols)[
                [
                    c
                    for c in df_linguistic.columns
                    if ("model_response" in c or "user_prompt" in c)
                    and (c not in ["model_response", "user_prompt"])
                ]
            ]
            .mean()
            .reset_index()
        )
    df = df.merge(
        df_linguistic[
            [id_col[dataset]]
            + [
                c
                for c in df_linguistic.columns
                if "model_response" in c
                or "user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
        ],
        on=id_col[dataset],
    )
    all_cols += [c for c in df.columns if "model_response" in c or "user_prompt" in c]

    df_beliefs = pd.read_pickle(
        f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
    )
    df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
    cols = [
        c
        for c in df_beliefs.columns
        if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
    ] + ["revealed_Gender"]
    if "human_Gender" in df_beliefs.columns:
        cols += ["human_Gender"]
    all_cols += cols
    cols.append(id_col[dataset])
    df = df.merge(df_beliefs[cols], on=id_col[dataset])

    # df[
    #     [
    #         c
    #         for c in all_cols
    #         if "model_response" in c or "user_prompt" in c
    #     ]
    # ] = minmax_scale(
    #     df[
    #         [
    #             c
    #             for c in all_cols
    #             if "model_response" in c or "user_prompt" in c
    #         ]
    #     ]
    # )

    cols = [
        # "accuracy",
        "benefits",
        "political",
        "legal",
        "medical",
        "salary",
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
    ] + [f"q_59_{v.replace(' ','').replace(',','')}" for v in options["q_59"]]

    # for demographic in demographics[dataset]:
    #     if not os.path.isfile(f"figures_regression/{dataset}_{demographic}_1.png"):
    #         filtered_df_demo = binarize_df(df, demographic)
    #         filtered_df_demo[demographic] = filtered_df_demo[demographic].astype(float)
    #         demo_cols = [
    #             c
    #             for c in all_cols
    #             if "model_response" in c or "user_prompt" in c or c == "model_name"
    #         ]
    #         mod = smf.ols(
    #             formula=f"{demographic} ~ {' + '.join(demo_cols)}",
    #             data=filtered_df_demo,
    #         )
    #         res = mod.fit()
    #         result_df = pd.read_html(
    #             res.summary().tables[1].as_html(), header=0, index_col=0
    #         )[0].reset_index()
    #         fig = plt.figure(figsize=(6.5, 5))
    #         ax = sns.barplot(
    #             result_df.loc[
    #                 (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
    #             ].sort_values(by="coef"),
    #             x="index",
    #             y="coef",
    #         )
    #         ax.tick_params(axis="x", labelrotation=90)
    #         fig.savefig(
    #             f"figures_regression/{dataset}_{demographic}_1.png", bbox_inches="tight"
    #         )
    #         plt.show()
    #         print(res.summary())

    #         filtered_df_demo = filtered_df_demo.loc[
    #             filtered_df_demo.duplicated(subset=["topic"], keep=False)
    #         ]

    #         demo_cols = [
    #             c
    #             for c in all_cols
    #             if "model_response" in c
    #             or "user_prompt" in c
    #             or c == "topic"
    #             or c == "model_name"
    #         ]
    #         mod = smf.ols(
    #             formula=f"{demographic} ~ {' + '.join(demo_cols)}",
    #             data=filtered_df_demo,
    #         )
    #         res = mod.fit()
    #         result_df = pd.read_html(
    #             res.summary().tables[1].as_html(), header=0, index_col=0
    #         )[0].reset_index()
    #         result_df["linguistic"] = ~result_df["index"].str.contains("topic")
    #         fig = plt.figure(figsize=(40, 5))
    #         ax = sns.barplot(
    #             result_df.loc[
    #                 (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
    #             ].sort_values(by="coef"),
    #             x="index",
    #             y="coef",
    #             hue="linguistic",
    #         )
    #         ax.tick_params(axis="x", labelrotation=90)
    #         fig.savefig(
    #             f"figures_regression/{dataset}_{demographic}_2.png", bbox_inches="tight"
    #         )
    #         plt.show()
    #         print(res.summary())

    for col in cols:
        filtered_df = df.loc[~df[col].isna()]
        if not os.path.isfile(f"figures_regression/{dataset}_{col}_correct_1.png"):
            demo_cols = [
                c
                for c in all_cols
                if "model_response" in c or "user_prompt" in c or c == "model_name"
            ]
            mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
            res = mod.fit()
            result_df = pd.read_html(
                res.summary().tables[1].as_html(), header=0, index_col=0
            )[0].reset_index()
            fig = plt.figure(figsize=(20, 5))
            ax = sns.barplot(
                result_df.loc[
                    (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
                ].sort_values(by="coef"),
                x="index",
                y="coef",
            )
            ax.tick_params(axis="x", labelrotation=90)
            fig.savefig(
                f"figures_regression/{dataset}_{col}_correct_1.png", bbox_inches="tight"
            )
            plt.show()
            print(res.summary())

        if "topic" in filtered_df:
            filtered_df = filtered_df.loc[
                filtered_df.duplicated(subset=["topic"], keep=False)
            ]
        if not os.path.isfile(f"figures_regression/{dataset}_{col}_correct_2.png"):
            demo_cols = [
                c
                for c in all_cols
                if "model_response" in c
                or "user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
            mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
            res = mod.fit()
            result_df = pd.read_html(
                res.summary().tables[1].as_html(), header=0, index_col=0
            )[0].reset_index()
            result_df["linguistic"] = ~result_df["index"].str.contains("topic")
            fig = plt.figure(figsize=(40, 5))
            ax = sns.barplot(
                result_df.loc[
                    (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
                ].sort_values(by="coef"),
                x="index",
                y="coef",
                hue="linguistic",
            )
            ax.tick_params(axis="x", labelrotation=90)
            fig.savefig(
                f"figures_regression/{dataset}_{col}_correct_2.png", bbox_inches="tight"
            )
            plt.show()
            print(res.summary())

        for demographic in [
            "age",
            "gender",
            "education",
            "ethnicity",
            "religion",
            "english",
            "marital",
            "political",
        ]:
            if not os.path.isfile(
                f"figures_regression/{dataset}_{col}_{demographic}_correct_3.png"
            ):
                demo_cols = [
                    c
                    for c in all_cols
                    if demographic in c.lower()
                    or "model_response" in c
                    or "user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]

                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                result_df["type"] = result_df["index"].str.extract("(topic)")
                result_df["type"].loc[
                    result_df["index"].str.contains(demographic)
                ] = "demographic"
                result_df["type"].loc[result_df["type"].isna()] = "linguistic"
                fig = plt.figure(figsize=(45, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                    hue="type",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_regression/{dataset}_{col}_{demographic}_correct_3.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())

In [ ]:
for dataset in [
    "cad_en",  # "chen",
    "prism",
]:
    all_cols = demographics[dataset]
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})

    for c in [qid for d in domains for qid in domain_qid_map[d] if d != "salary"]:
        df[c] = 1 * (df[c].str.lower() == questions_baseline_answers[c].lower())

    for c in domain_qid_map["salary"]:
        print(
            df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float),
            questions_baseline_answers[c],
        )
        df[c] = (
            df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)
            - questions_baseline_answers[c]
        )

    for domain in domains:
        df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
        if domain != "salary":
            df[domain] = df[domain] * 100

    for c in [
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
        "q_59",
    ]:
        if c == "q_53":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 0.5, "C": 1})
            df[c] = df[c].astype(float)
            df[c] = df[c] - questions_baseline_answers[c]
        elif c == "q_51":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 1})
            df[c] = df[c].astype(float)
            df[c] = df[c] - questions_baseline_answers[c]
        elif c == "q_58":
            df[c] = df[c].str.replace(" ", "")
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace(
                {
                    "1,1": 0,
                    "1,3": 0,
                    "3,1": 0,
                    "3,3": 0,
                    "2,2": 1,
                    "1,2": 0.5,
                    "1,4": 0.5,
                    "2,3": 0.5,
                    "2,1": 0.5,
                    "3,2": 0.5,
                    "3,4": 0.5,
                    "4,1": 0.5,
                    "4,3": 0.5,
                    "2,4": 1,
                    "4,2": 1,
                    "4,4": 1,
                }
            )
            df[c] = df[c].astype(float)
            df[c] = df[c] - questions_baseline_answers[c]
        elif c == "q_59":
            for v in options[c]:
                df[f"{c}_{v.replace(' ','').replace(',','')}"] = (
                    df[c].str.contains(v).astype(float)
                )
                df[f"{c}_{v.replace(' ','').replace(',','')}"] = (
                    df[f"{c}_{v.replace(' ','').replace(',','')}"]
                    - questions_baseline_answers[
                        f"{c}_{v.replace(' ','').replace(',','')}"
                    ]
                )
        else:
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].astype(float)
            df[c] = df[c] - questions_baseline_answers[c]
    df = df.drop(
        columns=[f"q_{i}" for i in range(50)]
        + ["q_59", "q_60"]
        + [f"q_{i}" for i in range(61, 211)]
    )

    df_linguistic = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    ).drop(
        columns=["s_neutral_model_response", "s_neutral_user_prompt"], errors="ignore"
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df_linguistic:
            df_linguistic[c] = df_linguistic[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    if dataset != "chen":
        if dataset == "prism":
            all_cols += ["model_name"]
        all_cols += ["topic"]
        group_cols = [id_col[dataset]] + all_cols
        df_linguistic = (
            df_linguistic.groupby(group_cols)[
                [
                    c
                    for c in df_linguistic.columns
                    if ("model_response" in c or "user_prompt" in c)
                    and (c not in ["model_response", "user_prompt"])
                ]
            ]
            .mean()
            .reset_index()
        )
    df = df.merge(
        df_linguistic[
            [id_col[dataset]]
            + [
                c
                for c in df_linguistic.columns
                if "model_response" in c
                or "user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
        ],
        on=id_col[dataset],
    )
    all_cols += [c for c in df.columns if "model_response" in c or "user_prompt" in c]

    df_beliefs = pd.read_pickle(
        f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
    )
    df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
    cols = [
        c
        for c in df_beliefs.columns
        if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
    ] + ["revealed_Gender"]
    if "human_Gender" in df_beliefs.columns:
        cols += ["human_Gender"]
    all_cols += cols
    cols.append(id_col[dataset])
    df = df.merge(df_beliefs[cols], on=id_col[dataset])

    # df[
    #     [
    #         c
    #         for c in all_cols
    #         if "_model_response" in c or "_user_prompt" in c
    #     ]
    # ] = minmax_scale(
    #     df[
    #         [
    #             c
    #             for c in all_cols
    #             if "_model_response" in c or "_user_prompt" in c
    #         ]
    #     ]
    # )

    cols = [
        # "accuracy",
        "benefits",
        "political",
        "legal",
        "medical",
        "salary",
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
    ] + [f"q_59_{v.replace(' ','').replace(',','')}" for v in options["q_59"]]

    for col in cols:
        filtered_df = df.loc[~df[col].isna()]
        if not os.path.isfile(f"figures_regression/{dataset}_{col}_baseline_1.png"):
            demo_cols = [
                c
                for c in all_cols
                if "model_response" in c or "user_prompt" in c or c == "model_name"
            ]
            mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
            res = mod.fit()
            result_df = pd.read_html(
                res.summary().tables[1].as_html(), header=0, index_col=0
            )[0].reset_index()
            fig = plt.figure(figsize=(20, 5))
            ax = sns.barplot(
                result_df.loc[
                    (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
                ].sort_values(by="coef"),
                x="index",
                y="coef",
            )
            ax.tick_params(axis="x", labelrotation=90)
            fig.savefig(
                f"figures_regression/{dataset}_{col}_baseline_1.png",
                bbox_inches="tight",
            )
            plt.show()
            print(res.summary())
        if "topic" in filtered_df:
            filtered_df = filtered_df.loc[
                filtered_df.duplicated(subset=["topic"], keep=False)
            ]
        if not os.path.isfile(f"figures_regression/{dataset}_{col}_baseline_2.png"):
            demo_cols = [
                c
                for c in all_cols
                if "model_response" in c
                or "user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
            mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
            res = mod.fit()
            result_df = pd.read_html(
                res.summary().tables[1].as_html(), header=0, index_col=0
            )[0].reset_index()
            result_df["linguistic"] = ~result_df["index"].str.contains("topic")
            fig = plt.figure(figsize=(40, 5))
            ax = sns.barplot(
                result_df.loc[
                    (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
                ].sort_values(by="coef"),
                x="index",
                y="coef",
                hue="linguistic",
            )
            ax.tick_params(axis="x", labelrotation=90)
            fig.savefig(
                f"figures_regression/{dataset}_{col}_baseline_2.png",
                bbox_inches="tight",
            )
            plt.show()
            print(res.summary())

        for demographic in [
            "age",
            "gender",
            "education",
            "ethnicity",
            "religion",
            "english",
            "marital",
            "political",
        ]:
            if not os.path.isfile(
                f"figures_regression/{dataset}_{col}_{demographic}_baseline_3.png"
            ):
                demo_cols = [
                    c
                    for c in all_cols
                    if demographic in c.lower()
                    or "model_response" in c
                    or "user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]

                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                result_df["type"] = result_df["index"].str.extract("(topic)")
                result_df["type"].loc[
                    result_df["index"].str.contains(demographic)
                ] = "demographic"
                result_df["type"].loc[result_df["type"].isna()] = "linguistic"
                fig = plt.figure(figsize=(45, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                    hue="type",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_regression/{dataset}_{col}_{demographic}_baseline_3.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())

In [ ]:
# for dataset in [
#     "cad_en",  # "chen",
#     "prism",
# ]:
#     for domain in ["benefits", "political", "salary", "legal", "medical"]:
#         df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
#         df = df.rename(columns={"label": "Gender"})
#         all_cols = deepcopy(demographics[dataset])
#         qrange = {"benefits": (61, 91), "political": (91,121), "salary": (121, 151), "legal": (151, 181), "medical": (181,211)}[domain]
#         if domain == "salary":
#             for c in [f"q_{i}" for i in range(*qrange)]:
#                 df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)
#         else:
#             for c in [f"q_{i}" for i in range(*qrange)]:
#                 df[c] = 1*(df[c].str.lower() == questions_correct_answers[c])

#         df = df.drop(
#         columns=[f"q_{i}" for i in range(0,211) if i not in range(*qrange)]
#         )
#         df = df.melt(id_vars=[c for c in df.columns if c not in [f'q_{qid}' for qid in range(*qrange)]], var_name="Question", value_name=domain)

#         df_linguistic = pd.read_pickle(
#             f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
#         ).drop(
#             columns=["s_neutral_model_response", "s_neutral_user_prompt"], errors="ignore"
#         )
#         for c in ["politeness_user_prompt", "politeness_model_response"]:
#             if c in df_linguistic:
#                 df_linguistic[c] = df_linguistic[c].replace(
#                     {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
#                 )
#         df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
#         if dataset != "chen":
#             if dataset == "prism":
#                 all_cols += ["model_name"]
#             all_cols += ["topic"]
#             group_cols = [id_col[dataset]] + all_cols
#             df_linguistic = (
#                 df_linguistic.groupby(group_cols)[
#                     [
#                         c
#                         for c in df_linguistic.columns
#                         if ("model_response" in c or "user_prompt" in c) and (c not in ["model_response", "user_prompt"])
#                     ]
#                 ]
#                 .mean()
#                 .reset_index()
#             )
#         df = df.merge(
#             df_linguistic[
#                 [id_col[dataset]]
#                 + [
#                     c
#                     for c in df_linguistic.columns
#                     if "model_response" in c
#                     or "user_prompt" in c
#                     or c == "topic"
#                     or c == "model_name"
#                 ]
#             ],
#             on=id_col[dataset],
#         )
#         all_cols += [
#             c for c in df.columns if "model_response" in c or "user_prompt" in c
#         ]

#         df_beliefs = pd.read_pickle(
#             f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
#         )
#         df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
#         cols = [
#             c
#             for c in df_beliefs.columns
#             if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
#         ] + ["revealed_Gender"]
#         if "human_Gender" in df_beliefs.columns:
#             cols += ["human_Gender"]
#         all_cols += cols
#         cols.append(id_col[dataset])
#         df = df.merge(df_beliefs[cols], on=id_col[dataset])

#         # df[
#         #     [
#         #         c
#         #         for c in all_cols
#         #         if "_model_response" in c or "_user_prompt" in c
#         #     ]
#         # ] = minmax_scale(
#         #     df[
#         #         [
#         #             c
#         #             for c in all_cols
#         #             if "_model_response" in c or "_user_prompt" in c
#         #         ]
#         #     ]
#         # )


#         filtered_df = df.loc[~df[domain].isna()]
#         if not os.path.isfile(f"figures_regression/{dataset}_{domain}_correct_1.png"):
#             demo_cols = [
#                 c
#                 for c in all_cols
#                 if "model_response" in c or "user_prompt" in c or c == "model_name"
#             ]
#             mod = smf.ols(formula=f"{domain} ~ {' + '.join(demo_cols)}", data=filtered_df)
#             res = mod.fit()
#             result_df = pd.read_html(
#                 res.summary().tables[1].as_html(), header=0, index_col=0
#             )[0].reset_index()
#             fig = plt.figure(figsize=(6.5, 5))
#             ax = sns.barplot(
#                 result_df.loc[
#                     (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
#                 ].sort_values(by="coef"),
#                 x="index",
#                 y="coef",
#             )
#             ax.tick_params(axis="x", labelrotation=90)
#             fig.savefig(
#                 f"figures_regression/{dataset}_{domain}_correct_1.png", bbox_inches="tight"
#             )
#             plt.show()
#             print(res.summary())

#         if "topic" in filtered_df:
#             filtered_df = filtered_df.loc[
#                 filtered_df.duplicated(subset=["topic"], keep=False)
#             ]
#         if not os.path.isfile(f"figures_regression/{dataset}_{domain}_correct_2.png"):
#             demo_cols = [
#                 c
#                 for c in all_cols
#                 if "model_response" in c
#                 or "user_prompt" in c
#                 or c == "topic"
#                 or c == "model_name"
#             ]
#             mod = smf.ols(formula=f"{domain} ~ {' + '.join(demo_cols)}", data=filtered_df)
#             res = mod.fit()
#             result_df = pd.read_html(
#                 res.summary().tables[1].as_html(), header=0, index_col=0
#             )[0].reset_index()
#             result_df["linguistic"] = ~result_df["index"].str.contains("topic")
#             fig = plt.figure(figsize=(40, 5))
#             ax = sns.barplot(
#                 result{column}_df.loc[
#                     (result_df["P>|t|"] < 0.05) & (result_df["index"] != "Intercept")
#                 ].sort_values(by="coef"),
#                 x="index",
#                 y="coef",
#                 hue="linguistic",
#             )
#             ax.tick_params(axis="x", labelrotation=90)
#             fig.savefig(
#                 f"figures_regression/{dataset}_{domain}_correct_2.png", bbox_inches="tight"
#             )
#             plt.show()
#             print(res.summary())

#         for demographic in [
#             "age",
#             "gender",
#             "education",
#             "ethnicity",
#             "religion",
#             "english",
#             "marital",
#             "political",
#         ]:
#             if not os.path.isfile(
#                 f"figures_regression/{dataset}_{domain}_{demographic}_correct_3.png"
#             ):
#                 demo_cols = [
#                     c
#                     for c in all_cols
#                     if demographic in c.lower()
#                     or "model_response" in c
#                     or "user_prompt" in c
#                     or c == "topic"
#                     or c == "model_name"
#                 ]

#                 mod = smf.ols(
#                     formula=f"{domain} ~ {' + '.join(demo_cols)}", data=filtered_df
#                 )
#                 res = mod.fit()
#                 result_df = pd.read_html(
#                     res.summary().tables[1].as_html(), header=0, index_col=0
#                 )[0].reset_index()
#                 result_df["type"] = result_df["index"].str.extract("(topic)")
#                 result_df["type"].loc[
#                     result_df["index"].str.contains(demographic)
#                 ] = "demographic"
#                 result_df["type"].loc[result_df["type"].isna()] = "linguistic"
#                 fig = plt.figure(figsize=(45, 5))
#                 ax = sns.barplot(
#                     result_df.loc[
#                         (result_df["P>|t|"] < 0.05)
#                         & (result_df["index"] != "Intercept")
#                     ].sort_values(by="coef"),
#                     x="index",
#                     y="coef",
#                     hue="type",
#                 )
#                 ax.tick_params(axis="x", labelrotation=90)
#                 fig.savefig(
#                     f"figures_regression/{dataset}_{domain}_{demographic}_correct_3.png",
#                     bbox_inches="tight",
#                 )
#                 plt.show()
#                 print(res.summary())